## 📝 **Notebook Overview: Hate Speech Classification with TACT**

This notebook tackles the Bengali hate speech classification task (Subtask 1A) by fine-tuning a `BanglaBERT` model.

### **Key Enhancements**
To improve upon the baseline, this notebook implements **Token-level Adversarial Contrastive Learning (TACT)**. This is achieved by:
1.  **Fast Gradient Method (FGM):** We introduce small, calculated perturbations to the token embeddings during training. These perturbations are designed to be adversarial—meaning they push the model towards making an incorrect prediction.
2.  **Consistency Regularization:** The model is trained to minimize the loss on *both* the original input and its adversarial counterpart. This forces the model to be robust and produce consistent outputs for similar inputs, leading to better generalization and higher accuracy.

This TACT approach is integrated with **Layer-wise Learning Rate Decay (LLRD)**, which applies smaller learning rates to the lower, more general layers of BERT and higher rates to the task-specific top layers, leading to more stable and effective fine-tuning.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

### **Step 1: Install Dependencies**

In [3]:
!pip install -q "transformers[torch]" datasets evaluate
!pip install -q git+https://github.com/csebuetnlp/normalizer
!pip install -q scikit-learn pandas matplotlib seaborn emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### **Step 2: Import Libraries and Mount Drive**

In [4]:
import os
import re
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt
import emoji

from google.colab import drive
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset, DatasetDict
from normalizer import normalize


2025-09-06 16:43:15.808312: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757176995.999959      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757176996.057083      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


### **Step 3: Define File Paths and Load Data**

In [5]:
# --- Configuration ---
# DRIVE_PATH = "/content/drive/MyDrive/Dataset"
#MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
MODEL_NAME = "csebuetnlp/banglabert"
#MODEL_NAME = "google/muril-base-cased"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- File Paths ---
TRAIN_FILE = "/kaggle/input/temp-1b2/blp25_hatespeech_subtask_1B_train.tsv"
DEV_FILE = "/kaggle/input/temp-1b2/blp25_hatespeech_subtask_1B_dev.tsv"
TEST_FILE = "/kaggle/input/temp-1b2/blp25_hatespeech_subtask_1B_dev_test.tsv"
SUBMISSION_FILE = "subtask_1B.tsv"
ZIP_SUBMISSION_FILE = "submission.zip"

# --- Load Datasets ---
try:
    df_train = pd.read_csv(TRAIN_FILE, sep='\t', dtype=str, keep_default_na=False)
    df_dev = pd.read_csv(DEV_FILE, sep='\t', dtype=str, keep_default_na=False)
    df_test = pd.read_csv(TEST_FILE, sep='\t', dtype=str, keep_default_na=False)
    print("Files loaded successfully!")
    print(f"Train shape: {df_train.shape}")
    print(f"Dev shape: {df_dev.shape}")
    print(f"Test shape: {df_test.shape}")
except FileNotFoundError as e:
    print(f"Error: {e}. Please check your file paths in Google Drive.")

display(df_train.head())

Files loaded successfully!
Train shape: (35522, 3)
Dev shape: (2512, 3)
Test shape: (2512, 2)


,id,text,label
0,147963,ধন্যবাদ বর্ডার গার্ড দেরকে এভাবে পাহারা দিতে হ...,None
1,214275,ছোটবেলায় অনেক কষ্ট করে কিছু গালাগালি শিখছিলাম...,None
2,849172,অতিরিক্ত এ নিজেকে বাদুর বানাইয়া ফেলছেন রে,Individual
3,821985,চিন ভারত রাশিয়া এই তিন দেশ এক থাকলে বিশ্বকে শা...,None
4,477288,এটার বিচার কে করবেযে বিচার করবে সেই তো হলো এই ...,Individual


### **Step 4: Text Preprocessing**
We use a robust cleaning function that normalizes Bengali unicode, demojizes emojis, and removes noise like URLs and HTML tags.

In [6]:
def refined_preprocess_text(text):
    text = normalize(text)
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'([.?!,])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Preprocessing text data...")
df_train['cleaned_text'] = df_train['text'].apply(refined_preprocess_text)
df_dev['cleaned_text'] = df_dev['text'].apply(refined_preprocess_text)
df_test['cleaned_text'] = df_test['text'].apply(refined_preprocess_text)

print("\n--- Sample of Cleaned Text ---")
for i in range(3):
    print(f"Original: {df_train['text'][i]}")
    print(f"Cleaned:  {df_train['cleaned_text'][i]}\n")
print(df_train.columns)
print(df_dev.columns)
print(df_test.columns)

Preprocessing text data...

--- Sample of Cleaned Text ---
Original: ধন্যবাদ বর্ডার গার্ড দেরকে এভাবে পাহারা দিতে হবে ভয় পেলে চলবে না তা না হলে আমাদের উপর হামলা করতে পারে
Cleaned:  ধন্যবাদ বর্ডার গার্ড দেরকে এভাবে পাহারা দিতে হবে ভয় পেলে চলবে না তা না হলে আমাদের উপর হামলা করতে পারে

Original: ছোটবেলায় অনেক কষ্ট করে কিছু গালাগালি শিখছিলাম ভাবছিলাম এরকম কোন সময় তা প্রয়োগ করব কিন্তু আফসোস কিছু বন্ধুদের স্ক্রিনশট এর ভয়ে তাও দিতে পারছিনা আপনার অবশ্যই বুঝে নিবেন
Cleaned:  ছোটবেলায় অনেক কষ্ট করে কিছু গালাগালি শিখছিলাম ভাবছিলাম এরকম কোন সময় তা প্রয়োগ করব কিন্তু আফসোস কিছু বন্ধুদের স্ক্রিনশট এর ভয়ে তাও দিতে পারছিনা আপনার অবশ্যই বুঝে নিবেন

Original: অতিরিক্ত এ নিজেকে বাদুর বানাইয়া ফেলছেন রে
Cleaned:  অতিরিক্ত এ নিজেকে বাদুর বানাইয়া ফেলছেন রে

Index(['id', 'text', 'label', 'cleaned_text'], dtype='object')
Index(['id', 'text', 'label', 'cleaned_text'], dtype='object')
Index(['id', 'text', 'cleaned_text'], dtype='object')


### **Step 5: Prepare Data for Hugging Face**
This involves label encoding, tokenizing the text, and creating `Dataset` objects.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Label Encoding
# Remove rows with NaN values in 'label' column
df_train_cleaned = df_train.dropna(subset=['label']).copy()
df_dev_cleaned = df_dev.dropna(subset=['label']).copy()

labels_list = sorted(df_train_cleaned['label'].unique())
# label2id = { "None": 0, "Religious Hate": 1, "Sexism": 2, "Political Hate": 3, "Profane": 4, "Abusive": 5, }

label2id = {
    'Society': 0,
    'Organization': 1,
    'None': 2,
    'Individual': 3,
    'Community': 4
}



id2label = {v: k for k, v in label2id.items()}
NUM_LABELS = len(labels_list)

print(f"Label to ID mapping: {label2id}")

df_train_cleaned['labels'] = df_train_cleaned['label'].map(label2id)
df_dev_cleaned['labels'] = df_dev_cleaned['label'].map(label2id)

# Create Hugging Face Datasets
train_dataset = Dataset.from_pandas(df_train_cleaned[['id','cleaned_text', 'labels']])
dev_dataset = Dataset.from_pandas(df_dev_cleaned[['id','cleaned_text', 'labels']])
test_dataset = Dataset.from_pandas(df_test[['id','cleaned_text']]) # Test set does not have labels

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': dev_dataset,
    'test': test_dataset
})

# Tokenization Function
def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], truncation=True, max_length=256)

print("\nTokenizing datasets...")
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# Data Collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\n--- Final Prepared Datasets ---")
print(tokenized_datasets)

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Label to ID mapping: {'Society': 0, 'Organization': 1, 'None': 2, 'Individual': 3, 'Community': 4}

Tokenizing datasets...


Map:   0%|          | 0/35522 [00:00<?, ? examples/s]

Map:   0%|          | 0/2512 [00:00<?, ? examples/s]

Map:   0%|          | 0/2512 [00:00<?, ? examples/s]


--- Final Prepared Datasets ---
DatasetDict({
    train: Dataset({
        features: ['id', 'cleaned_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 35522
    })
    validation: Dataset({
        features: ['id', 'cleaned_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2512
    })
    test: Dataset({
        features: ['id', 'cleaned_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2512
    })
})


### **Step 6: TACT Implementation (FGM + Custom Trainer)**
Here's the core of the new approach. We define an `FGM` class to handle the adversarial attack on embeddings and a `TACTTrainer` that overrides the default loss computation to include the adversarial training step.

In [8]:
class FGM:
    """
    Fast Gradient Method (FGM) for adversarial training.
    Adds a perturbation to the model's embeddings.
    """
    def __init__(self, model):
        self.model = model
        self.backup = {}

    def attack(self, epsilon=1.0, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                assert name in self.backup
                param.data = self.backup[name]
        self.backup = {}

class TACTTrainer(Trainer):
    """
    Custom Trainer that incorporates FGM for TACT.
    The total loss is the sum of the original loss and the adversarial loss.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fgm = FGM(self.model)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # --- Standard Forward Pass ---
        outputs = model(**inputs)
        loss = outputs.loss

        # --- Adversarial Training Step ---
        # Only perform adversarial training if the model is in training mode
        if model.training:
            loss.backward(retain_graph=True)

            # 1. FGM Attack
            # NOTE: name might change for other models, inspect model.electra.embeddings.word_embeddings.named_parameters()
            self.fgm.attack(emb_name='electra.embeddings.word_embeddings')

            # 2. Compute loss on adversarial example
            adv_outputs = model(**inputs)
            adv_loss = adv_outputs.loss

            # 3. Restore original embeddings
            self.fgm.restore(emb_name='electra.embeddings.word_embeddings')

            # 4. Combine losses
            loss = loss + adv_loss

        return (loss, outputs) if return_outputs else loss

print("TACT components (FGM, TACTTrainer) are defined.")

TACT components (FGM, TACTTrainer) are defined.


In [9]:
# --- Class-weighted CrossEntropy & Weighted TACT Trainer (minimal add-on) ---
import numpy as np
import torch
from torch import nn
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights from training labels
y_train = np.array(tokenized_datasets["train"]["labels"])
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float)

# Subclass TACTTrainer to apply weighted CE while keeping the adversarial step
class WeightedTACTTrainer(Trainer):
    def __init__(self, *args, class_weights=None, epsilon=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.epsilon = float(epsilon)
        self._ce = None

    def _get_embed_weight(self, model):
        for name, p in model.named_parameters():
            if "embeddings.word_embeddings.weight" in name:
                return p
        for name, p in model.named_parameters():
            if "embeddings.word_embeddings" in name:
                return p
        return None

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")

        if self._ce is None:
            w = self.class_weights.to(model.device) if self.class_weights is not None else None
            self._ce = nn.CrossEntropyLoss(weight=w)

        # 1) base forward
        outputs = model(**inputs)
        logits = outputs.get("logits")
        base_loss = self._ce(logits, labels)

        # 2) adversarial forward
        adv_loss = 0.0
        emb_param = self._get_embed_weight(model)
        if model.training and emb_param is not None and self.epsilon > 0:  # <-- fixed here
            grad = torch.autograd.grad(base_loss, emb_param, retain_graph=True, allow_unused=True)[0]
            if grad is not None:
                norm = torch.norm(grad)
                if torch.isfinite(norm) and norm > 0:
                    r_adv = self.epsilon * grad / norm
                    emb_param.data.add_(r_adv)
                    adv_outputs = model(**inputs)
                    adv_logits = adv_outputs.get("logits")
                    adv_loss = self._ce(adv_logits, labels)
                    emb_param.data.sub_(r_adv)

        loss = base_loss + (adv_loss if isinstance(adv_loss, torch.Tensor) else 0.0)
        return (loss, outputs) if return_outputs else loss


print("WeightedTACTTrainer ready (uses class-weighted CrossEntropy).")

WeightedTACTTrainer ready (uses class-weighted CrossEntropy).


### **Step 7: LLRD Optimizer Setup**
We keep the Layer-wise Learning Rate Decay optimizer, as it is a powerful technique for fine-tuning Transformer models and complements TACT well.

In [10]:
# def create_llrd_optimizer(model, learning_rate=2e-5, layer_decay=0.95):
#     optimizer_grouped_parameters = []
#     no_decay = ["bias", "LayerNorm.weight"]
#     num_layers = model.electra.config.num_hidden_layers

#     # Classifier layers
#     optimizer_grouped_parameters.extend([
#         {
#             "params": [p for n, p in model.classifier.named_parameters() if not any(nd in n for nd in no_decay)],
#             "weight_decay": 0.01, "lr": learning_rate
#         },
#         {
#             "params": [p for n, p in model.classifier.named_parameters() if any(nd in n for nd in no_decay)],
#             "weight_decay": 0.0, "lr": learning_rate
#         }
#     ])

#     # Transformer layers with decay
#     for i in range(num_layers - 1, -1, -1):
#         layer_lr = learning_rate * (layer_decay ** (num_layers - 1 - i))
#         layer_params = [
#             {
#                 "params": [p for n, p in model.electra.encoder.layer[i].named_parameters() if not any(nd in n for nd in no_decay)],
#                 "weight_decay": 0.01, "lr": layer_lr
#             },
#             {
#                 "params": [p for n, p in model.electra.encoder.layer[i].named_parameters() if any(nd in n for nd in no_decay)],
#                 "weight_decay": 0.0, "lr": layer_lr
#             }
#         ]
#         optimizer_grouped_parameters.extend(layer_params)

#     # Embedding layer
#     embedding_lr = learning_rate * (layer_decay ** num_layers)
#     optimizer_grouped_parameters.extend([
#         {
#             "params": [p for n, p in model.electra.embeddings.named_parameters() if not any(nd in n for nd in no_decay)],
#             "weight_decay": 0.01, "lr": embedding_lr
#         },
#         {
#             "params": [p for n, p in model.electra.embeddings.named_parameters() if any(nd in n for nd in no_decay)],
#             "weight_decay": 0.0, "lr": embedding_lr
#         }
#     ])

#     return torch.optim.AdamW(optimizer_grouped_parameters, lr=learning_rate)

# print("LLRD optimizer function defined.")

In [11]:
def create_llrd_optimizer(model, learning_rate=2e-5, layer_decay=0.95):
    optimizer_grouped_parameters = []
    no_decay = ["bias", "LayerNorm.weight"]

    # Detect backbone automatically (bert, roberta, electra, etc.)
    if hasattr(model, "roberta"):
        base_model = model.roberta
    elif hasattr(model, "bert"):
        base_model = model.bert
    elif hasattr(model, "electra"):
        base_model = model.electra
    elif hasattr(model, "xlm_roberta"):
        base_model = model.xlm_roberta
    else:
        base_model = model.base_model  # fallback

    num_layers = base_model.config.num_hidden_layers

    # ----- Classifier head -----
    optimizer_grouped_parameters.extend([
        {
            "params": [p for n, p in model.classifier.named_parameters()
                       if not any(nd in n for nd in no_decay)],
            "weight_decay": 0.01,
            "lr": learning_rate,
        },
        {
            "params": [p for n, p in model.classifier.named_parameters()
                       if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ])

    # ----- Transformer encoder layers -----
    for i in range(num_layers - 1, -1, -1):
        layer_lr = learning_rate * (layer_decay ** (num_layers - 1 - i))
        optimizer_grouped_parameters.extend([
            {
                "params": [p for n, p in base_model.encoder.layer[i].named_parameters()
                           if not any(nd in n for nd in no_decay)],
                "weight_decay": 0.01,
                "lr": layer_lr,
            },
            {
                "params": [p for n, p in base_model.encoder.layer[i].named_parameters()
                           if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": layer_lr,
            },
        ])

    # ----- Embedding layer -----
    embedding_lr = learning_rate * (layer_decay ** num_layers)
    optimizer_grouped_parameters.extend([
        {
            "params": [p for n, p in base_model.embeddings.named_parameters()
                       if not any(nd in n for nd in no_decay)],
            "weight_decay": 0.01,
            "lr": embedding_lr,
        },
        {
            "params": [p for n, p in base_model.embeddings.named_parameters()
                       if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": embedding_lr,
        },
    ])

    return torch.optim.AdamW(optimizer_grouped_parameters, lr=learning_rate)

print("✅ LLRD optimizer function defined (works with XLM-R, BERT, RoBERTa, ELECTRA, etc.).")


✅ LLRD optimizer function defined (works with XLM-R, BERT, RoBERTa, ELECTRA, etc.).


### **Step 8: Model Training**
We configure the training arguments and instantiate our new `TACTTrainer` to begin training.

In [12]:
# # model = AutoModelForSequenceClassification.from_pretrained(
# #     MODEL_NAME,
# #     num_labels=NUM_LABELS,
# #     id2label=id2label,
# #     label2id=label2id
# # ).to(DEVICE)

# # def compute_metrics(pred):
#     labels = pred.label_ids
#     preds = pred.predictions.argmax(-1)
#     precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
#     acc = accuracy_score(labels, preds)
#     # Additional: weighted F1 for reference
#     f1_weighted = precision_recall_fscore_support(labels, preds, average='weighted')[2]
#     return {
#         'accuracy': acc,
#         'macro_precision': precision,
#         'macro_recall': recall,
#         'macro_f1': f1,
#         'weighted_f1': f1_weighted,
#     }

# # training_args = TrainingArguments(
# #     output_dir="./results",
# #     num_train_epochs=5, # Adversarial training can benefit from more epochs
# #     per_device_train_batch_size=16,
# #     per_device_eval_batch_size=16,
# #     learning_rate=2e-5, # Base LR for LLRD
# #     weight_decay=0.01,
# #     warmup_ratio=0.1,
# #     eval_strategy="epoch",
# #     save_strategy="epoch",
# #     load_best_model_at_end=True,
# #     metric_for_best_model='macro_f1', # F1-score is a more robust metric here
# #     greater_is_better=True,
# #     fp16=torch.cuda.is_available(),
# #     report_to="none",
# #     save_total_limit=1,
# # )

# # optimizer = create_llrd_optimizer(model, learning_rate=training_args.learning_rate) #adamw valo so use it

# # trainer = WeightedTACTTrainer(
# #     model=model,
# #     args=training_args,
# #     train_dataset=tokenized_datasets["train"],
# #     eval_dataset=tokenized_datasets["validation"],
# #     tokenizer=tokenizer,
# #     data_collator=data_collator,
# #     compute_metrics=compute_metrics,
#     class_weights=class_weights,
# #     optimizers=(optimizer, None) # Pass the custom optimizer
# # )

# # print("Starting model training with TACT and LLRD...")
# # trainer.train()

# # print("\nTraining finished.")

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import torch

# Load model with correct head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,       # your 6-class task
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # <-- FIX: reinitialize classifier if mismatch
).to(DEVICE)

# Metrics
# def compute_metrics(pred):
#     labels = pred.label_ids
#     preds = pred.predictions.argmax(-1)
#     precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
#     acc = accuracy_score(labels, preds)
#     return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# def compute_metrics(pred):
#     labels = pred.label_ids
#     preds = pred.predictions.argmax(-1)
#     precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
#     acc = accuracy_score(labels, preds)
#     # Additional: weighted F1 for reference
#     f1_weighted = precision_recall_fscore_support(labels, preds, average='weighted')[2]
#     return {
#         'accuracy': acc,
#         'macro_precision': precision,
#         'macro_recall': recall,
#         'macro_f1': f1,
#         'weighted_f1': f1_weighted,
#     }
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    weighted_f1 = precision_recall_fscore_support(labels, preds, average='weighted')[2]
    return {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",   # <-- correct param name
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=1,
)

# Optimizer (LLRD/AdamW)
optimizer = create_llrd_optimizer(model, learning_rate=training_args.learning_rate)

# # Trainer
# trainer = TACTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["validation"],
#     tokenizer=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
#     optimizers=(optimizer, None),
# )

trainer = WeightedTACTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    optimizers=(optimizer, None) # Pass the custom optimizer
)


print("Starting model training with TACT and LLRD...")
trainer.train()
print("\nTraining finished.")

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_36/212970793.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTACTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Starting model training with TACT and LLRD...


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,2.400000,0.994249,0.588774,0.515746,0.653016,0.535374,0.613146
2,2.127000,0.956598,0.616242,0.519046,0.645045,0.543366,0.645269
3,1.889200,0.956511,0.672373,0.553693,0.667983,0.585505,0.694772


### **Step 9: Final Evaluation**
We evaluate the best model checkpoint on the development set.

In [14]:
print("Evaluating the best model on the validation set...")
eval_results = trainer.evaluate()

print("\n--- Final Evaluation Results ---")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

Evaluating the best model on the validation set...



--- Final Evaluation Results ---
eval_loss: 1.4144
eval_accuracy: 0.6835
eval_macro_precision: 0.5492
eval_macro_recall: 0.6319
eval_macro_f1: 0.5800
eval_weighted_f1: 0.6980
eval_runtime: 5.5650
eval_samples_per_second: 451.3920
eval_steps_per_second: 28.2120
epoch: 10.0000


In [15]:
print(tokenized_datasets["test"])

Dataset({
    features: ['id', 'cleaned_text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2512
})


### **Step 10: Prediction and Submission**
Generate predictions on the test set and create the `submission.zip` file.

In [16]:
print("Making predictions on the test set...")
test_dataset_formatted = tokenized_datasets["test"].remove_columns([col for col in ['labels', 'label'] if col in tokenized_datasets["test"].column_names])
ids_2 = tokenized_datasets["test"]['id']
test_predictions = trainer.predict(test_dataset_formatted)
preds = test_predictions.predictions
#print(ids_2)

Making predictions on the test set...


In [17]:
print(test_predictions)

PredictionOutput(predictions=array([[-2.0097656 , -0.89160156,  6.2617188 , -1.1572266 , -2.5917969 ],
       [-1.3154297 ,  2.9316406 , -1.5439453 ,  1.4853516 , -0.96728516],
       [-3.0292969 , -2.9589844 ,  3.2304688 ,  4.8789062 , -2.3632812 ],
       ...,
       [-0.9814453 , -3.171875  ,  1.6992188 ,  3.9199219 , -1.8789062 ],
       [-2.7714844 , -0.6352539 ,  6.0585938 ,  0.24047852, -3.1738281 ],
       [-1.1191406 ,  0.08325195,  5.328125  , -2.1386719 , -2.5332031 ]],
      dtype=float32), label_ids=None, metrics={'test_runtime': 5.8745, 'test_samples_per_second': 427.611, 'test_steps_per_second': 26.726})


In [18]:
# def standardize_list(data):
#     """
#     Standardize a list/array of numbers (Z-score normalization).

#     Parameters:
#         data (list or np.ndarray): A list/array of numerical values.

#     Returns:
#         list: Standardized values with mean 0 and std dev 1.
#     """
#     # Convert to numpy array (handles both list and array inputs)
#     arr = np.array(data, dtype=float)

#     if arr.size == 0:
#         raise ValueError("Input list/array is empty.")

#     mean = arr.mean()
#     std_dev = arr.std()

#     if std_dev == 0:
#         raise ValueError("Standard deviation is zero; cannot standardize.")

#     return ((arr - mean) / std_dev).tolist()

In [19]:
LABEL2ID = {
    'Society': 0,
    'Organization': 1,
    'None': 2,
    'Individual': 3,
    'Community': 4
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
logits_list_2 = [list(row) for row in preds]
#model_name = 'Custom'

In [20]:
output_predict_file = os.path.join(training_args.output_dir, f"subtask_1B_muril.tsv")
if trainer.is_world_process_zero():
    with open(output_predict_file, "w") as writer:
        #logger.info(f"***** Predict results *****")
        writer.write("id\tlogits\tmodel\n")
        for index, item in enumerate(logits_list_2):
            # item = id2l[item]
            #writer.write(f"{ids[index]}\t{item}\t{model_name}\n")
            writer.write(f"{ids_2[index]}\t{' '.join(map(str, item))}\t{MODEL_NAME}\n")